In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
import re

from config import RESSERENO_ROOT

In [2]:
# pd.set_option('display.max_rows', None)

In [2]:
path = RESSERENO_ROOT / "data" / "fall2024_data collection"

## Functions and lists for free association and valence task

In [3]:
# includes all the folders of the different years
folders = [f"{path}/Stella1oAnno", f"{path}/Canal2oAnno", f"{path}/Lombardi3oAnno"]

# function to associate the name of the year to all the files based on their initial characters 
def year_extraction(filename):
    if filename.startswith('0'):
       return'first_year'
    if filename.startswith('LC'):
        return 'second_year'
    if filename.startswith('LL'):
        return 'third_year'


# create a list that has the filename, its ordinal number, the year and that creates also the participant_id
filenames = []

for folder_path in folders:
    for filename in os.listdir(folder_path):
        year = year_extraction(filename)
        if filename.endswith(".xlsx"):
            match = re.search(r'(\d{3})', filename)
            if match:
                filenames.append({
                    'number': int(match.group(1)), 
                    'year' : year, 
                    'filename' : filename, 
                    'file_path': os.path.join(folder_path, filename),
                    'participant_id': f"{year}_{match.group(1)}",
                    })

# make the list into a df, sorted by year and number 
filenames = pd.DataFrame(filenames).sort_values(by = ['year', 'number']).reset_index(drop=True)

# takes only the 'participant_id' and stores it into a variable
participant_ids = filenames['participant_id']

In [4]:
# all the 41 keywords used in the experiment
keywords = ['voto', 'compito', 'noia', 'stress', 'biologia', 'comportamento', 'scoperta', 'università', 'neuroscienze', 'grafico', 'cognizione', 'emozione', 'sfida', 'creatività', 'scienza', 'statistica', 'relazione', 'lavoro', 'tecnologia', 'equazione', 'fisica', 'psicologia','personalità', 'terapia', 'esercitazione', 'mente', 'numeri', 'futuro', 'frustrazione', 'matematica', 'esame', 'benessere', 'modello',  'professore', 'ansia', 'laurea', 'conoscenza', 'curiosità', 'scuola', 'insegnante', 'studente']

# function to iterate over the files, read them and store them into a df
def file_iteration(folders, sheet_name):
    for file_path, filename in zip(filenames['file_path'], filenames['filename']):
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        yield df, filename

# function to check which keyword is missing from the file, describe the findings and store them in an excel file 
def missing_cue_words(df, keywords, filename, save_path):
    missing_words = []

    df['cue word'] = df['cue word'].str.strip()

    duplicated_keywords = df[df['cue word'].duplicated(keep=False)]['cue word'].unique().tolist()

    missing_keywords = [keyword for keyword in keywords if keyword not in df['cue word'].values]

    if len(df) != len(keywords) or duplicated_keywords or missing_keywords:
        missing_words.append(f'filename: {filename}, length: {len(df)}, missing words: {missing_keywords}, duplicated words: {duplicated_keywords}')

    with open(save_path, "a") as file:
        for line in missing_words:
            file.write(line + "\n")

In [23]:
# function to create lists 
def build_lists(df, filename, dict_lists, col_names):

    # loop to create lists and append them to the dictionary
    for col_name in col_names:
        dict_lists[col_name].append(df[col_name].reset_index(drop=True))
    
    # since there are missing words, let's get the length of the df (number of cue words)
    num_rows = len(df)

    # 
    participant_id = dict(zip(filenames['filename'], filenames['participant_id'])).get(filename)

    # append the participant_id to the dictionary multiplying it by the number of cue words present in the df
    dict_lists['participant_id'].append(pd.Series([participant_id] * num_rows))
    
    # creates a df for lists
    combined_lists = {
    key: pd.concat(series_list, ignore_index=True)
    for key, series_list in dict_lists.items()
    }

    # sort the df by 'cue word' at first and then by 'filename'
    lists_df = pd.DataFrame(combined_lists).sort_values(by=['cue word', 'participant_id']).reset_index(drop=True)

    # rename 'cue word' to 'cue_word'
    lists_df.rename(columns={"cue word": "cue_word"}, inplace=True)

    # Keep only the first occurrence of each duplicated cue word
    lists_df = lists_df[~lists_df.duplicated(subset=['participant_id', 'cue_word'], keep=False)]

    lists_df = lists_df.map(lambda x: x.strip().lower() if isinstance(x, str) else x)

    # insert as first column the 'filename' and then all the other columns
    cols = ['participant_id'] + [col for col in lists_df.columns if col != 'participant_id']
    lists_df = lists_df[cols]

    # return the df
    return lists_df

# Task associazioni

In [6]:
output_path = "/Users/emma/projects/cognoscolab/rassereno/out/formatting/psychology_BA_2024"

In [24]:
# names of the columns in the association task
col_names_associations = ["cue word", "association_1", "association_2", "association_3"]

# creates a dictionary for each column, except for 'cue word', used as index
associations_map = defaultdict(
    lambda: {col: [] for col in col_names_associations if col != "cue word"}
)

# created a dictionary of lists to store the association, it creates a default value for keys that don't exist yet
associations_lists = defaultdict(list)

# save path for the .txt file of the missing keywords
save_path = f"{output_path}/missing_keywords_associations_task.txt"

# closes the file so that it writes the new information
open(save_path, "w").close()

# call of iteration function of the associations task sheet
for df, filename in file_iteration(folders, "Task_1parte_Associazioni"):

    df = df[col_names_associations][10:].dropna(how='all')
    
    # call the function to create lists and save them in csv file
    associations_df = build_lists(df, filename, associations_lists, col_names_associations)
    
    # call of function missing keywords in the association task
    missing_cue_words(df, keywords, filename, save_path)

# Task valenza

In [25]:
# names of the columns for the valence task 
col_names_valence = ['cue word', 'valence_cue_word', 'association_1', 'valence_association_1', 'association_2', 'valence_association_2', 'association_3', 'valence_association_3']

valence_scores_map = defaultdict(
    lambda: {col: [] for col in col_names_valence if col != "cue word"}
)

# created a dictionary of lists to store the associations and valence scores, it creates a default value for keys that don't exists yet
valence_lists = defaultdict(list)

# save path for the .txt file of the missing keywords
save_path = f'{output_path}/missing_keywords_valence_task.txt'

# closes the file so that it writes the new information
open(save_path, "w").close()

# call of iteration function of the associations task sheet
for df, filename in file_iteration(folders, 'Task_2parte_Valenza'):

    # takes the values, the data concerning the valence scores
    df = df.iloc[8:, range(8)].dropna(how='all')
    
    # assigns the column names to the columns 
    df.columns = col_names_valence

    # call the function to create lists and save them in csv file
    valence_df = build_lists(df, filename, valence_lists, col_names_valence)
    
    # call of function missing keywords in the valence task
    missing_cue_words(df, keywords, filename, save_path)

In [26]:
valence_columns = [
    'valence_cue_word',
    'valence_association_1',
    'valence_association_2',
    'valence_association_3'
]
for col in valence_columns:
    valence_df[col] = pd.to_numeric(valence_df[col], errors='coerce')

for col in valence_df.columns:
    if valence_df[col].dtype == object:
        valence_df[col] = valence_df[col].str.strip()

valence_df = valence_df.replace([int(0)], np.nan)

In [ ]:
# get the percentage of missing data (NaN values) for each participants
missing_values = (
    valence_df
    .set_index("participant_id")
    .isna()
    .mean(axis=1)
    .groupby(level=0)
    .mean()
)

# make a list of participants to drop based of the percentage of missing data (more than the 1/3 of data)
to_drop = missing_values[missing_values > 1/3].index
print("Participants to drop:", len(to_drop), list(to_drop))

# create a df where we remove all the participants of the list to_drop
associations_valence_df = valence_df[~valence_df["participant_id"].isin(to_drop)]

Participants to drop: 2 ['second_year_039', 'second_year_062']


In [38]:
associations_valence_df.to_csv(f'{output_path}/associations_valence_2024.csv', index=False)
print(f"Number of cleaned participants: {len(associations_valence_df['participant_id'].unique())}")

Number of cleaned participants: 166


## Differences between two associations and valence  dfs

In [ ]:
def compare_df(df1, df2):
    if df1.equals(df2) == False:
        comparison = (df1 == df2) | (df1.isna() & df2.isna())

        mismatched_mask = ~comparison.all(axis=1)
        mismatched_rows_df1 = df1[mismatched_mask]
        mismatched_rows_df2 = df2[mismatched_mask]

        print(f"Mismatched rows in df1: {mismatched_rows_df1}")
        print(f"Mismatched rows in df2: {mismatched_rows_df2}")

        return 'False: the two dataframes are different'
    else:
        return 'True: the two dataframes are equal'

In [37]:
columns_to_compare = ["participant_id", "cue_word", "association_1", "association_2", "association_3"]

associations_df = associations_df[~associations_df["participant_id"].isin(to_drop)]
valence_df = valence_df[~valence_df["participant_id"].isin(to_drop)]

df1_subset = associations_df[columns_to_compare].reset_index(drop=True)
df2_subset = valence_df[columns_to_compare].reset_index(drop=True)

df1_subset = df1_subset.combine_first(df2_subset)
df2_subset = df2_subset.combine_first(df1_subset)

compare_df(df1_subset, df2_subset)

'True: the two dataframes are equal'

# Parte dati demografici e questionari

## Dati demografici

In [39]:
# creates the lists for gender and age and assigns to them the values corresponding to certain excel cells
age = []
gender = []

for df, filename in file_iteration(folders, 'Dati_Demografici'):
    age.append([float(df.iloc[5, 1])])
    gender.append([df.iloc[9, 1]])

# transform them into df with a column name
age_df = pd.DataFrame(age, columns=['age'])
gender_df = pd.DataFrame(gender, columns=['gender'])

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## Questionari

In [40]:
# function to parse the values found in the excel cells
def parse_value(x):
    # check if the values are NaN
    if pd.isna(x):
        return None
    try:
        # if it's already a number (int or float) return it directly
        if isinstance(x, (int, float)):
            return int(x) if float(x).is_integer() else float(x)
        
        # if the value is a string we take the first part of the string and normalize it
        token = str(x).strip().split()[0]
        
        # converting the token of the the string to float only if it's a decimal, if not we transform it into a integer
        number = float(token)
        return int(number) if number.is_integer() else number
    except:
        return None

In [41]:
# function that allows to extract the data from the questionnaire 
def extract_questionnaire_data(folders, sheet_name, row_range, column_index, headers):
    # creates empty list to store the values found in a determined position in the excel file
    data = []

    # extracts the values and it cleans them from unnecessary text (for example in the likert scores descriptions) 
    for df, filename in file_iteration(folders, sheet_name):

        # extract the values from specific excel cells
        values = df.iloc[row_range, column_index].values
        
        # calls the parsing function to get the cleaned data from the cells values
        cleaned_values = [parse_value(x) for x in values]

        # append the cleaned data to the list 
        data.append(cleaned_values)
        
    # makes the list into a df with specific headers
    return pd.DataFrame(data, columns=headers)

In [42]:
# Math abilities

math_abilities = extract_questionnaire_data(
    folders = folders, 
    sheet_name='Questionari', 
    row_range= range(5, 12, 2), 
    column_index=1,
    headers = ['materie STEM', 'meterie umanistiche', 'matematica e fisica', 'programmazione']
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [43]:
# Need for cognition

need_cognition = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(20, 56, 2),
    column_index=1,
    headers=[f'need_cognition_{i}' for i in range(1, 19)]
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [44]:
# Big 5 reduced

big_5 = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(64, 84, 2),
    column_index=1,
    headers=[f'big_5_{i}' for i in range(1, 11)]
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [47]:
# MAS-IT

mas_it = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(91, 137, 2),
    column_index=1,
    headers=[f'mas_it_{i}' for i in range(1, 24)]
)

correspondence_order = [
    'mas_it_1', 'mas_it_3', 'mas_it_6', 'mas_it_7', 'mas_it_10', 'mas_it_18', 'mas_it_19',
    'mas_it_21', 'mas_it_23', 'mas_it_2', 'mas_it_4', 'mas_it_5', 'mas_it_8', 'mas_it_11',
    'mas_it_13', 'mas_it_14', 'mas_it_22', 'mas_it_9', 'mas_it_12', 'mas_it_15',
    'mas_it_16', 'mas_it_17', 'mas_it_20'
]

# reorder columns based on the original order
mas_it_reordered = mas_it[correspondence_order]

# # name columns to match the original names
mas_it_reordered.columns = mas_it.columns

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [48]:
mas_it_df = pd.concat([participant_ids, mas_it_reordered], axis=1)
mas_it_cleaned = mas_it_df[~mas_it_df["participant_id"].isin(to_drop)].reset_index(drop=True)
mas_it_cleaned.to_csv(f'{output_path}/mas_it_2024.csv', index=False)

In [49]:
# STAI: State Anxiety

state_anxiety = extract_questionnaire_data(
    folders=folders,
    sheet_name='Questionari',
    row_range=range(143, 157, 2),
    column_index=1,
    headers=[f'state_anxiety_{i}' for i in range(1, 8)]
)

/Users/emma/projects/cognoscolab/myvenv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## Merge and export to .csv all the dfs

In [50]:
# concatenates all the df related to demographic data and questionnaire response + the participants ids
all_df = pd.concat([participant_ids, age_df, gender_df, math_abilities, need_cognition, big_5, mas_it, state_anxiety], axis=1)

all_df = all_df[~all_df["participant_id"].isin(to_drop)].reset_index(drop=True)

# exports the concatenated dfs into an csv file
all_df.to_csv(f'{output_path}/all_questionnaires_task_2024.csv', index=False)

# Breve testo

In [54]:
# these are the strings corresponding to the instructions given by the researchers
instruction_1 = 'Scriva adesso un breve testo di circa 250 parole relativamente a come si sente quando si trova ad affrontare le discipline STEM.'
instruction_2 = '"Quando affronto le discipline STEM…"'
# unnecessary string
punctuation = '…'

# creates a list with all the texts of all the participant extracted in the excel files
text = []

for df, filename in file_iteration(folders, 'breve_testo'):
    df = df.dropna().reset_index(drop=True)
    participant_text = " ".join(df.iloc[:, 0].astype(str))
    # creates a string with '.' at the end and removes alla the instruction strings
    participant_text = f"{participant_text.replace(instruction_1, '').replace(instruction_2, '').replace(punctuation, '').strip()}."
    # appends the text for each participant in the list
    text.append(participant_text)

# converts the text into a df 
texts = pd.DataFrame(text, columns=['text'])
# concatenates the text and the participants ids
texts_df = pd.concat([participant_ids, texts], axis=1)

texts_df = texts_df[~texts_df["participant_id"].isin(to_drop)].reset_index(drop=True)

# export the df into a csv file
texts_df.to_csv(f'{output_path}/text_task_2024.csv', index=False)